# 🤖 OpsMind AI — Enterprise Agentic Operations Platform

**Stack:** LangGraph · LangChain · Ollama · FastAPI · Scikit-learn · XGBoost · TensorFlow · ChromaDB · Streamlit · OAuth2 · JWT · RBAC

## Setup Order:
1. Runtime → Change runtime type → **T4 GPU** (free)
2. Run cells **in order** (Cell 1 → 2 → 3 → ... → 8)
3. Cell 6 gives you a **public ngrok URL** for the Streamlit UI

### Login credentials:
| Username | Password | Role |
|----------|----------|------|
| gowtham | gowtham2026 | Super Admin |
| admin | admin123 | Admin |
| analyst | analyst123 | Analyst |
| viewer | viewer123 | Viewer |

In [ ]:
# ─── CELL 1: Clone repo & install dependencies ───────────────
# Push your code to GitHub first, then:
# !git clone https://github.com/YOUR_USERNAME/opsmind-ai.git
# %cd opsmind-ai

# OR manually copy the files from this project
import os
os.makedirs('/content/opsmind', exist_ok=True)
%cd /content/opsmind

!pip install -q \
    langchain==0.3.7 \
    langchain-community==0.3.7 \
    langchain-core==0.3.15 \
    langgraph==0.2.45 \
    langchain-ollama==0.2.0 \
    fastapi==0.115.4 \
    uvicorn[standard]==0.32.0 \
    python-jose[cryptography]==3.3.0 \
    passlib[bcrypt]==1.7.4 \
    scikit-learn==1.5.2 \
    xgboost==2.1.2 \
    chromadb==0.5.20 \
    sentence-transformers==3.3.0 \
    streamlit==1.40.0 \
    plotly==5.24.1 \
    pydantic==2.9.2 \
    pyngrok==7.2.0 \
    requests==2.32.3 \
    faiss-cpu==1.9.0

print('✅ Dependencies installed!')

In [ ]:
# ─── CELL 2: Install & Start Ollama ─────────────────────────
!curl -fsSL https://ollama.ai/install.sh | sh

import subprocess, threading, time

def run_ollama():
    subprocess.Popen(['ollama', 'serve'],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

threading.Thread(target=run_ollama, daemon=True).start()
time.sleep(5)
print('✅ Ollama server started')

# Pull model — llama3.2 is ~2GB, fast on Colab free tier
!ollama pull llama3.2
print('✅ llama3.2 model ready!')

In [ ]:
# ─── CELL 3: Write all source files ─────────────────────────
# After pushing to GitHub, this cell is just:
#   import sys; sys.path.insert(0, '/content/opsmind-ai')
# 
# For now, paste all the code files from the project here
# (auth/auth_core.py, ml/models.py, agents/graph.py, api/main.py)
# See the GitHub repo README for the full file list.

import sys
sys.path.insert(0, '/content/opsmind')
print('✅ Path configured')

In [ ]:
# ─── CELL 4: Train ML Models ────────────────────────────────
from ml.models import train_all_models

print('🏋️ Training ML models (Isolation Forest + XGBoost + LSTM)...')
ad, fp, lstm = train_all_models(save=True)
print('✅ All ML models trained!')

In [ ]:
# ─── CELL 5: Initialize ChromaDB Knowledge Base ─────────────
import chromadb, os
from sentence_transformers import SentenceTransformer

os.makedirs('data/chroma_db', exist_ok=True)
client = chromadb.PersistentClient(path='data/chroma_db')
collection = client.get_or_create_collection('opsmind_runbooks')

runbooks = [
    'HIGH CPU USAGE RUNBOOK: When CPU exceeds 85%, check top processes. Scale horizontally. Enable CPU throttling.',
    'HIGH MEMORY USAGE RUNBOOK: Memory above 90% triggers GC. Analyze heap dumps. Check connection pools.',
    'HIGH ERROR RATE RUNBOOK: Error rate above 15% — check upstream deps. Enable circuit breaker. Rollback if needed.',
    'HIGH LATENCY RUNBOOK: Latency above 500ms — check DNS, CDN, load balancer. Enable response compression.',
    'INCIDENT RESPONSE: P1 = escalate within 5min. Create war room. Document all actions. Post-mortem in 48h.',
    'K8S SCALING RUNBOOK: Verify node capacity before HPA triggers. Check PodDisruptionBudget. Monitor rollout.',
]

encoder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = encoder.encode(runbooks).tolist()
collection.add(documents=runbooks, ids=[f'rb_{i}' for i in range(len(runbooks))], embeddings=embeddings)
print(f'✅ ChromaDB initialized with {len(runbooks)} operational runbooks!')

In [ ]:
# ─── CELL 6: Start FastAPI Backend ──────────────────────────
import subprocess, threading, time, requests

def run_api():
    subprocess.Popen(
        ['python', '-m', 'uvicorn', 'api.main:app', '--host', '0.0.0.0', '--port', '8000'],
        stdout=open('api.log', 'w'), stderr=open('api_err.log', 'w')
    )

threading.Thread(target=run_api, daemon=True).start()
time.sleep(10)

try:
    r = requests.get('http://localhost:8000/health')
    print('✅ FastAPI running:', r.json())
except Exception as e:
    print('❌ API startup issue. Check api_err.log:', e)
    !tail -20 api_err.log

In [ ]:
# ─── CELL 7: Start Streamlit + ngrok Public URL ──────────────
import subprocess, threading, time
from pyngrok import ngrok

def run_streamlit():
    subprocess.Popen(
        ['streamlit', 'run', 'streamlit_app/app.py',
         '--server.port=8501', '--server.headless=true',
         '--browser.gatherUsageStats=false'],
        stdout=open('streamlit.log', 'w'), stderr=open('streamlit_err.log', 'w')
    )

threading.Thread(target=run_streamlit, daemon=True).start()
time.sleep(12)

# Optional: set ngrok authtoken for stable URL
# ngrok.set_auth_token('YOUR_NGROK_TOKEN')

ui_url  = ngrok.connect(8501)
api_url = ngrok.connect(8000)

print('=' * 55)
print('🌐  STREAMLIT UI :', ui_url)
print('📡  API SWAGGER  :', str(api_url) + '/docs')
print('=' * 55)

In [ ]:
# ─── CELL 8: Quick API Test ──────────────────────────────────
import requests

BASE = 'http://localhost:8000'

# Login
resp = requests.post(f'{BASE}/auth/token',
    data={'username': 'gowtham', 'password': 'gowtham2026', 'grant_type': 'password'})
token = resp.json()['access_token']
headers = {'Authorization': f'Bearer {token}'}
print('✅ Login role:', resp.json()['role'])

# Metrics
m = requests.get(f'{BASE}/metrics/live', headers=headers).json()
print('📊 CPU:', round(m['cpu_usage'], 1), '| Memory:', round(m['memory_usage'], 1))

# Anomaly on injected data
m_anomaly = requests.get(f'{BASE}/metrics/live?inject_anomaly=true', headers=headers).json()
anomaly_payload = {k: v for k, v in m_anomaly.items() if k not in ('timestamp', 'service')}
anomaly_payload['service'] = 'test'
result = requests.post(f'{BASE}/anomaly/detect', json=anomaly_payload, headers=headers).json()
print('🚨 Anomaly detected:', result['is_anomaly'], '| Score:', result['anomaly_score'])

# Feature importance
fi = requests.get(f'{BASE}/predict/feature-importance', headers=headers).json()
print('📊 Top feature:', max(fi['feature_importance'], key=fi['feature_importance'].get))

print('\n✅ All tests passed!')

In [ ]:
# ─── CELL 9: Run Agent Pipeline (requires Ollama) ────────────
from agents.graph import run_agent

print('🧠 Running LangGraph multi-agent pipeline...')
print('This may take 1-3 minutes depending on Ollama speed')
print()

result = run_agent(
    task='Analyze current system health and generate a comprehensive incident report',
    user_role='analyst'
)

print('=' * 60)
print(result.get('report', 'No report generated'))
print('=' * 60)
print('Iterations:', result.get('iterations'))
print('Anomaly:', result.get('anomaly'))
print('Failure risk:', result.get('prediction', {}).get('risk_level', 'N/A'))